In [19]:
(require sicp)

### Exercise 1.1

Below is a sequence of expressions. What is the result printed by the interpreter in response to each expression? Assume that the sequence is to be evaluated in the order in which it is presented.

In [20]:
10

10

In [21]:
(+ 5 3 4)

12

In [22]:
(- 9 1)

8

In [23]:
(/ 6 2)

3

In [24]:
(+ (* 2 4) ( - 4 6))

6

In [25]:
(define a 3)
a

3

In [26]:
(define b (+ a 1))
b

4

In [28]:
(+ a b (* a b))

19

In [29]:
(= a b)

#f

In [30]:
(if (and (> b a) (< b (* a b)))
    b
    a)

4

In [ ]:
(cond ((= a 4) 6)
      ((= b 4) (+ 6 7 a))
      (else 25))

16

In [32]:
(+ 2 (if (> b a) b a))

6

In [33]:
( * (cond ((> a b) a)
          ((< a b) b)
          (else -1))
    (+ a 1))

16

---

### Exercise 1.2

Translate the following expression into prefix form:

$\frac{5 + 4 + (2 - (3 - (6 + \frac{4}{5})))}{3(6 - 2)(2 - 7)}$

In [1]:
(/ (+ 5 4 ( - 2(- 3 (+ 6 (/ 4 5)))))
   (* 3 (- 6 2 ) (- 2 7)))

-37/150

---

### Exercise 1.3

Define a procedure that takes three numbers as arguments and returns the sum of the squares of the two larger numbers.

In [ ]:
(define (<= a b) (not (> a b)))

(define (max a b c)
  (cond ((and (<= a b ) (<= a c)) (+ (* b b) (* c c)))
        ((and (<= b a ) (<= b c)) (+ (* a a) (* c c)))
        (else (+ (* a a) (* b b)))))

(max 5 2 2)


29

---

### Exercise 1.4

Observe that our model of evaluation allows for combinations whose operators are compound expressions. Use this observation to describe the behavior of the following procedure:

```lisp
(define (a-plus-abs-b a b) 
  ((if (> b 0) + -) a b))
```

This procedure first selects the operator `+` if `b` is greater than 0 and `-` otherwise. It then adds `b` to `a` or subtracts `b` from `a` and returns the result.

---

### Exercise 1.5

Ben Bitdiddle has invented a test to determine whether the interpreter he is faced with is using applicative-order evaluation or normal-order evaluation. He defines the following two procedures:

In [ ]:
(define (p) (p))
(define (test x y) 
  (if (= x 0) 0 y))

Then he evaluates the expression:

In [10]:
(test 0 (p))

user break
  context...:
   body of top-level


What behavior will Ben observe with an interpreter that uses applicative-order evaluation?  What behavior will he observe with an interpreter that uses normal-order evaluation? Explain your answer.
(Assume that the evaluation rule for the special form `if` is the same whether the interpreter is using normal or applicative order: The predicate expression is evaluated ﬁrst, and the result determines whether to evaluate the consequent or the alternative expression.)

**applicative order**

Evaluate arguments before passing them to the function --> infinite recursion because `(p)` calls itself.

**normal order**:

Evaluate arguments as needed
`(test 0 (p))` via substitution of `test` becomes `(if (= 0 0) 0 (p))`
and since `(= 0 0)` is true, it returns `0` immediately
(because `if` is a special operation)




Example: Newton's method for iterative approximation of square roots.

In [4]:
(define (sqrt-iter guess x)
  (if (good-enough? guess x)
    guess 
    (sqrt-iter (improve guess x) x))) ; recursion

(define (improve guess x)
  (average guess (/ x guess))) 
  
(define (average x y)
  (/ (+ x y) 2))

(define (good-enough? guess x)
  (< (abs (- (square guess) x)) 0.001))

(define (square x)
  (* x x))

(define (sqrt x)
  (sqrt-iter 1.0 x))

(sqrt 9)

3.00009155413138

---

### Exercise 1.6

Alyssa P. Hacker doesn't see why if needs to be provided as a special form. "Why can't I just define it as an ordinary procedure in terms of cond?" she asks. Alyssa's friend Eva Lu Ator claims this can indeed be done, and she defines a new version of if:

```lisp
(define (new-if predicate then-clause else-clause)
  (cond (predicate then-clause)
        (else else-clause)))
```

Eva demonstrates the program for Alyssa:

```
(new-if (= 2 3) 0 5) 5
(new-if (= 1 1) 0 5) 0
```

Delighted, Alyssa uses new-if to rewrite the square-root program:

```lisp
(define (sqrt-iter guess x)
  (new-if (good-enough? guess x)
          guess
          (sqrt-iter (improve guess x) x)))
```

What happens when Alyssa attempts to use this to compute square roots? Explain.

Infinite recursion because, unlike with the special operator `if`, the interpreter evaluates `sqrt-iter` before applying the condition.

---

### Exercise 1.7

The good-enough? test used in computing square roots will not be very effective for finding the square roots of very small numbers. Also, in real computers, arithmetic operations are almost always performed with limited precision. This makes our test inadequate for very large numbers. Explain these statements, with examples showing how the test fails for small and large numbers. An alternative strategy for implementing good-enough? is to watch how guess changes from one iteration to the next and to stop when the change is a very small fraction of the guess. Design a square-root procedure that uses this kind of end test. Does this work better for small and large numbers?

For small numbers the criterion is too lenient.

In [ ]:
(sqrt 0.000001) ; should be 0.001

0.031260655525445276

For large numbers the criterion is too strict, making the algorithm inefficient. If we hit the limits of floating point precision the algorithm will recurse infinitely.

In [ ]:
(sqrt 12736734343421)

user break
  context...:
   eval:15:0: body of top-level
   eval:12:0: good-enough?
   eval:1:0: sqrt-iter


In [48]:
(define (sqrt-iter guess x)
  (if (< (improve-ratio (improve guess x) guess) 0.001)
    guess
    (sqrt-iter (improve guess x) x)))

(define (improve-ratio guess-im guess)
  (/ (abs (- guess-im guess)) guess))

(define (abs x)
  (if (< x 0) 
      (* -1 x)
      x))

Test that this handles very small and large numbers.

In [50]:
(sqrt 12736734343421)

3569445.489224337

In [51]:
(sqrt 0.000001)

0.0010005538710539446

---

### Exercise 1.8
Newton’s method for cube roots is based on the fact that ify is an approximation to the cube root of x, then a better approximation is given by the value $\frac{x/y2 + 2y}{3}$ .

Use this formula to implement a cube-root procedure analogous to the square-root procedure. (In Section 1.3.4 we will see how to implement Newton’s method in general as an abstraction of these square-root and cube-root procedures.)

In [55]:
(define (improve guess x)
  (/ (+ 
       (/ x (* guess guess))
       (* 2 guess))
     3))


In [ ]:
(sqrt 27); should be 3

3.001274406506175

---

## Exercise 1.9

Each of the following two procedures deﬁnes a method for adding two positive integers in terms of the procedures `inc`, which increments its argument by 1, and `dec`, which decrements its argument by 1.
```lisp
(define (+ a b)
  (if (= a 0) b (inc (+ (dec a) b))))

(define (+ a b)
  (if (= a 0) b (+ (dec a) (inc b))))
```
 Using the substitution model, illustrate the process generated by each procedure in evaluating (+ 4 5). Are these processes iterative or recursive?

Procedure 1:

```
(+ 4 5)
(inc (+ (dec 4) 5))
(inc (+ 3 5))
(inc (inc (+ (dec 3) 5 )))
...
(inc (inc (inc (inc (+ (dec 1) 5)))))
(inc (inc (inc (inc 5))))
9
```

This is process is recursive

Procedure 2:

```
(+ 4 5)
(+ (dec 4) (inc 5))
(+ 3 6)
(+ (dec 3) (inc 6))
(+ 2 7)
(+ (dec 2) (inc 7))
(+ 1 8)
(+ (dec 1) (inc 8))
(+ 0 9)
9
```
This process is iterative

The key distinction: iterative processes have constant space (state is in the parameters), while recursive processes require space proportional to the input (state is in the deferred operations).

---

### Exercise 1.10

The following procedure computes a mathematical function called Ackermann’s function.

In [2]:
(define (A x y)
  (cond ((= y 0) 0) 
        ((= x 0) (* 2 y))
        ((= y 1) 2)
        (else (A (- x 1) (A x (- y 1))))))

What are the values of the following expressions?
```
(A 2 4)
(A 3 3)
```


In [ ]:
(A 1 10)

1024

In [7]:
(A 2 4)

65536

In [8]:
(A 3 3)

65536

#### (A 1 10)

`(A 1 n)` computes $2^n$. When `x=1`, the function keeps calling itself with decreasing `y` until `y=1`, building up a chain of `(A 0 ...)` calls which each double their argument:

```
(A 1 10)
(A 0 (A 1 9))
(A 0 (A 0 (A 1 8)))
...
(A 0 (A 0 (A 0 (A 0 (A 0 (A 0 (A 0 (A 0 (A 0 (A 1 1))))))))))
(A 0 (A 0 (A 0 (A 0 (A 0 (A 0 (A 0 (A 0 (A 0 2)))))))))  ; (A 1 1) = 2
(A 0 (A 0 (A 0 (A 0 (A 0 (A 0 (A 0 (A 0 4))))))))
(A 0 (A 0 (A 0 (A 0 (A 0 (A 0 (A 0 8)))))))
...
1024  ; 2^10
```

#### (A 2 4)

```
(A 2 4)
(A 1 (A 2 3))
(A 1 (A 1 (A 2 2)))
(A 1 (A 1 (A 1 (A 2 1))))
(A 1 (A 1 (A 1 2)))           ; (A 2 1) = 2 (y=1 → 2)
(A 1 (A 1 4))                 ; (A 1 2) = 2² = 4
(A 1 16)                      ; (A 1 4) = 2⁴ = 16
65536                         ; (A 1 16) = 2¹⁶ = 65536
```

`(A 2 n)` computes **tetration** — a tower of 2s with height n: $^n2 = 2^{2^{2^{...}}}$

#### (A 3 3)

```
(A 3 3)
(A 2 (A 3 2))
(A 2 (A 2 (A 3 1)))
(A 2 (A 2 2))                 ; (A 3 1) = 2 (y=1 → 2)
(A 2 4)                       ; (A 2 2) = 2² = 4
65536                         ; (A 2 4) = 2^2^2^2 = 65536
```

`(A 3 n)` computes **pentation** — repeated tetration.

Consider the following procedures, where A is the procedure deﬁned above:  

In [3]:
(define (f n)
  (A 0 n)) 

(define (g n)
  (A 1 n))

(define (h n)
  (A 2 n))

(define (k n)
  (* 5 n n))

Give concise mathematical deﬁnitions for the functions computed by the procedures f, g, and h for positive integer values of n. For example, $(k n)$ computes $5n^2$.

`(f n)` computes $2*n$
`(g n)` computes $2^n$
`(h n)` comutes ${}^{n}2$ which is $\underbrace{2^{2^{2^{...}}}}_{n}$

---

## Exercise 1.11

A function $f$ is defined by the rule that

$$
f(n) = \begin{cases}
n & \text{if } n < 3, \\
f(n-1) + 2f(n-2) + 3f(n-3) & \text{if } n \geq 3.
\end{cases}
$$

Write a procedure that computes $f$ by means of a recursive process. Write a procedure that computes $f$ by means of an iterative process.

In [14]:
(define (fr n)
    (if (< n 3) n
        (+ (fr (- n 1)) (* 2 (fr (- n 2))) (* 3 (fr (- n 3))))))

In [15]:
(fr 10)

1892

In [18]:
(define (fi n)
  (define (iter a b c count)
    (if (= count 0)
        a
        (iter (+ a (* 2 b) (* 3 c)) a b (- count 1))))
  (if (< n 3)
      n
      (iter 4 2 1 (- n 3))))


In [19]:
(fi 10)

1892